# P6-1 · Bài 8a · SFT và PEFT

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
# một dòng trong train.jsonl, dạng prompt/completion hội thoại
{"prompt": [{"role": "system", "content": "Trích xuất ý định và mã đơn. Chỉ trả JSON."},
            {"role": "user", "content": "Tôi muốn đổi size áo đơn DH1024"}],
 "completion": [{"role": "assistant", "content": "{\"intent\": \"đổi hàng\", \"order\": \"DH1024\"}"}]}

{'prompt': [{'role': 'system', 'content': 'Trích xuất ý định và mã đơn. Chỉ trả JSON.'}, {'role': 'user', 'content': 'Tôi muốn đổi size áo đơn DH1024'}], 'completion': [{'role': 'assistant', 'content': '{"intent": "đổi hàng", "order": "DH1024"}'}]}


## Bài tập 4: Định dạng dữ liệu hội thoại {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, pandas as pd

# file mẫu 2 dòng để chạy thử (dữ liệu thật: vài trăm tới vài nghìn dòng)
pd.DataFrame({"text": ["Đơn DH1024 giao sai màu ghế, mình cần đổi gấp trước thứ 7.", "Cho hỏi bàn trang điểm còn hàng không?"],
              "json_label": ['{"intent": "đổi hàng", "order": "DH1024", "product": "ghế", "urgent": true}',
                             '{"intent": "hỏi hàng", "order": null, "product": "bàn trang điểm", "urgent": false}']}
             ).to_csv("emails.csv", index=False)

SYSTEM = "Trích xuất ý định, mã đơn, sản phẩm và mức khẩn từ email khách hàng. Chỉ trả JSON."
df = pd.read_csv("emails.csv")
with open("train.jsonl", "w", encoding="utf-8") as f:
    for _, r in df.iterrows():
        f.write(json.dumps({
            "prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": r["text"]}],
            "completion": [{"role": "assistant", "content": r["json_label"]}],
        }, ensure_ascii=False) + "\n")
print(open("train.jsonl", encoding="utf-8").readline())

{"prompt": [{"role": "system", "content": "Trích xuất ý định, mã đơn, sản phẩm và mức khẩn từ email khách hàng. Chỉ trả JSON."}, {"role": "user", "content": "Đơn DH1024 giao sai màu ghế, mình cần đổi gấp trước thứ 7."}], "completion": [{"role": "assistant", "content": "{\"intent\": \"đổi hàng\", \"order\": \"DH1024\", \"product\": \"ghế\", \"urgent\": true}"}]}



In [ ]:
# Cần mạng: tải tokenizer Qwen từ Hugging Face (chạy trên Colab)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
sample = json.loads(open("train.jsonl", encoding="utf-8").readline())
print(tok.apply_chat_template(sample["prompt"] + sample["completion"], tokenize=False))

## Bài tập 5: Nhãn chỉ tính loss trên câu trả lời {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn.functional as F
phan = [("system", "Bạn là trợ lý của shop nội thất ."), ("user", "Bàn trang điểm có mấy ngăn kéo ?"),
        ("assistant", "Dạ , bàn có 3 ngăn kéo và 1 gương LED . <eos>")]
vocab = {w: i for i, w in enumerate(sorted({w for _, t in phan for w in t.split()} | {"<system>", "<user>", "<assistant>"}))}
# TODO: input_ids (thêm thẻ vai trò trước mỗi phần), labels; loss_tat_ca, loss_tra_loi, so_token_tinh

In [ ]:
input_ids, labels = [], []
for vai, text in phan:
    input_ids.append(vocab[f"<{vai}>"]); labels.append(-100)
    for w in text.split():
        input_ids.append(vocab[w]); labels.append(vocab[w] if vai == "assistant" else -100)
input_ids, labels = torch.tensor(input_ids), torch.tensor(labels)
torch.manual_seed(0)
logits = torch.randn(len(input_ids), len(vocab))
loss_tat_ca = F.cross_entropy(logits, input_ids)
loss_tra_loi = F.cross_entropy(logits, labels, ignore_index=-100)
so_token_tinh = (labels != -100).sum()
print(f"{len(input_ids)} token, tính loss trên {so_token_tinh}: loss mọi token {loss_tat_ca:.3f}, chỉ câu trả lời {loss_tra_loi:.3f}")

33 token, tính loss trên 13: loss mọi token 3.896, chỉ câu trả lời 3.796


## Bài tập 6: Bộ nhớ GPU cho ba cách fine-tune {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
def bo_nho(n_ty, cach):
    # TODO: trả về GB (1 GB = 1e9 byte)
    return 0.0

for n in (1.5, 7, 14):
    print(n, {c: round(bo_nho(n, c), 1) for c in ("full", "lora", "qlora")})

1.5 {'full': 0.0, 'lora': 0.0, 'qlora': 0.0}
7 {'full': 0.0, 'lora': 0.0, 'qlora': 0.0}
14 {'full': 0.0, 'lora': 0.0, 'qlora': 0.0}


In [ ]:
def bo_nho(n_ty, cach):
    lora = 0.01 * n_ty * 16
    return {"full": 16 * n_ty, "lora": 2 * n_ty + lora, "qlora": 0.5 * n_ty + lora}[cach]

print(f"{'model':>6} {'full':>7} {'LoRA':>7} {'QLoRA':>7}")
for n in (1.5, 7, 14):
    print(f"{n:>5}B {bo_nho(n, 'full'):>6.1f}G {bo_nho(n, 'lora'):>6.1f}G {bo_nho(n, 'qlora'):>6.1f}G")

 model    full    LoRA   QLoRA
  1.5B   24.0G    3.2G    1.0G
    7B  112.0G   15.1G    4.6G
   14B  224.0G   30.2G    9.2G


## Bài tập 7: Làm sạch dữ liệu SFT {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import re
data = [
    [{"role": "user", "content": "Bàn có mấy màu?"}, {"role": "assistant", "content": "Dạ có 3 màu: trắng, hồng, gỗ."}],
    [{"role": "user", "content": "Bàn có mấy màu? "}, {"role": "assistant", "content": "Dạ có 3 màu: trắng, hồng, gỗ."}],
    [{"role": "user", "content": "Giao hàng mất bao lâu?"}, {"role": "assistant", "content": ""}],
    [{"role": "user", "content": "Có trả góp không?"}, {"role": "assistant", "content": "Dạ [TÊN NGÂN HÀNG] hỗ trợ trả góp 0%."}],
    [{"role": "assistant", "content": "Chào bạn!"}, {"role": "user", "content": "Shop ở đâu?"}],
    [{"role": "user", "content": "Bảo hành bao lâu?"}, {"role": "assistant", "content": "Dạ khung bàn bảo hành 12 tháng ạ."}],
    [{"role": "user", "content": "Mô tả bàn"}, {"role": "assistant", "content": "Bàn đẹp. " * 60}],
    [{"role": "user", "content": "Có giao tỉnh không?"}, {"role": "assistant", "content": "Dạ shop giao toàn quốc ạ."},
     {"role": "user", "content": "Phí bao nhiêu?"}, {"role": "assistant", "content": "Dạ từ 30.000đ tùy cân nặng."}],
]

def hop_le(mau):
    # TODO
    return True

In [ ]:
def hop_le(mau):
    vai = [m["role"] for m in mau]
    if vai != ["user", "assistant"] * (len(vai) // 2) or len(vai) % 2:
        return False
    for m in mau[1::2]:
        t = m["content"].strip()
        if not t or len(t) > 400 or re.search(r"\[[A-ZÀ-Ỹ ]+\]|TODO", t):
            return False
    return True

def khoa(mau):
    return " || ".join(" ".join(m["content"].lower().split()) for m in mau)

sach, da_co = [], set()
for m in data:
    if hop_le(m) and khoa(m) not in da_co:
        da_co.add(khoa(m)); sach.append(m)
print(f"giữ {len(sach)}/{len(data)} mẫu")

giữ 3/8 mẫu
